## Pendefinisian File Geometry untuk Desa di Kabupaten Sijunjung

In [1]:
import geopandas as gpd

# Path ke file shp
shp_path = "Kelurahan_Sijunjung.shp"

# Baca shapefile
gdf = gpd.read_file(shp_path)

# Tampilkan seluruh atribut (kolom)
print("Daftar Kolom:")
print(gdf.columns)

# Tampilkan 5 baris pertama
print("\nSample Data:")
print(gdf.head())

# Jika ingin menampilkan semua baris
print("\nSeluruh Data Atribut:")
print(gdf)

Daftar Kolom:
Index(['KDEBPS', 'KDEPUM', 'KDPBPS', 'KDPKAB', 'KDPPUM', 'LUASWH', 'TIPADM',
       'WADMKC', 'WADMKD', 'WADMKK', 'WADMPR', 'LUAS', 'geometry'],
      dtype='object')

Sample Data:
  KDEBPS         KDEPUM KDPBPS KDPKAB KDPPUM  LUASWH  TIPADM        WADMKC  \
0   None  13.03.06.2004   None  13.03     13     0.0     1.0   Kamang Baru   
1   None  13.03.04.2008   None  13.03     13     0.0     1.0     Sijunjung   
2   None  13.03.10.2001   None  13.03     13     0.0     1.0       Kupitan   
3   None  13.03.08.2006   None  13.03     13     0.0     1.0      Koto Vii   
4   None  13.03.07.2003   None  13.03     13     0.0     1.0  Lubuak Tarok   

          WADMKD     WADMKK          WADMPR        LUAS  \
0        Aie Amo  Sijunjung  Sumatera Barat  127.627821   
1      Aie Angek  Sijunjung  Sumatera Barat   69.595853   
2  Batu Manjulur  Sijunjung  Sumatera Barat   18.486907   
3     Bukit Bual  Sijunjung  Sumatera Barat    4.546669   
4   Buluah Kasok  Sijunjung  Sumatera Bar

## Filtering Data dan Ekspor ke Excel

In [2]:
gdf_new = gdf[['WADMKD','KDEPUM']]

In [3]:
gdf_new.to_excel("BahanJoinSHP.xlsx", index=False)

## Penggabungan Data Prelist SE dari MatchaPro

In [1]:
import pandas as pd
import glob
import os

# Path folder tempat file Excel berada
folder_path = "Prelist SE Kabupaten Sijunjung"  # ganti dengan path folder kamu

# Ambil semua file .xlsx dalam folder
file_list = glob.glob(os.path.join(folder_path, "*.xlsx"))

# Gabungkan semua file
df_list = [pd.read_excel(file) for file in file_list]
df_final = pd.concat(df_list, ignore_index=True)

# Tampilkan hasil
print(df_final.shape)
print(df_final.head())

# Simpan hasil gabungan
df_final.to_excel("PrelistSEGabungan.xlsx", index=False)

print("Berhasil digabung dan disimpan sebagai PrelistSEGabungan.xlsx")

(38854, 18)
    IDSBR              Nama Usaha                           Alamat  Kdprov  \
0  219814    KOPDES KAMPUNG SURAU  Kampung Surau, Kec. Pl. Punjung      13   
1  219815    KOPPAS MARGA SIKABAU    Ps. Sikabau, Kec. Pl. Punjung      13   
2  219812  KOPPAS KENAG SEI DAREH      Sei Dareh, Kec. Pl. Punjung      13   
3  219813     KOPINKRA PRIMA JAYA     P. Punjung, Kec. Pl. Punjung      13   
4  219830     KOPONTREN MTI PULAI              Pulai, Kec. Sitiung      13   

   Kdkab  Kdkec  Kddesa          Nmprov      Nmkab Nmkec Nmdesa  \
0      4    NaN     NaN  SUMATERA BARAT  SIJUNJUNG   NaN    NaN   
1      4    NaN     NaN  SUMATERA BARAT  SIJUNJUNG   NaN    NaN   
2      4    NaN     NaN  SUMATERA BARAT  SIJUNJUNG   NaN    NaN   
3      4    NaN     NaN  SUMATERA BARAT  SIJUNJUNG   NaN    NaN   
4      4    NaN     NaN  SUMATERA BARAT  SIJUNJUNG   NaN    NaN   

    Status Perusahaan Kegiatan Usaha Skala Usaha Sumber Data  \
0  Salah Kode Wilayah            NaN          UB    

## Transformasi Data Prelist SE dengan Agregat pada Level Desa

In [3]:
import pandas as pd
import re

df = pd.read_excel("PrelistSEGabungan.xlsx", sheet_name="Fix")
# ==========================
# 1. Ekstrak KBLI dari kegiatan_usaha
# ==========================
df['KBLI'] = df['Kegiatan Usaha'].str.extract(r'KBLI:\s*(\d+)')

# ==========================
# 2. Group kolom wilayah
# ==========================
group_cols = [
    'Kdprov', 'Kdkab', 'Kdkec', 'Kddesa',
    'Nmkab', 'Nmkec', 'Nmdesa'
]

# ==========================
# 3. Buat pivot tabel (KBLI jadi kolom)
# ==========================
pivot = pd.pivot_table(
    df,
    index=group_cols,
    columns='KBLI',
    aggfunc='size',
    fill_value=0
)

# ==========================
# 4. Tambahkan jumlah usaha per desa
# ==========================
pivot['jumlah_usaha'] = pivot.sum(axis=1)

# ==========================
# 5. Reset index supaya jadi dataframe biasa
# ==========================
result = pivot.reset_index()

print(result.head())

KBLI  Kdprov  Kdkab  Kdkec  Kddesa      Nmkab        Nmkec  \
0         13      4   50.0     1.0  SIJUNJUNG  KAMANG BARU   
1         13      4   50.0     2.0  SIJUNJUNG  KAMANG BARU   
2         13      4   50.0     3.0  SIJUNJUNG  KAMANG BARU   
3         13      4   50.0     4.0  SIJUNJUNG  KAMANG BARU   
4         13      4   50.0     5.0  SIJUNJUNG  KAMANG BARU   

KBLI                  Nmdesa  01111  01117  01121  ...  95291  95299  96111  \
0              SUNGAI LANSEK      2      0      0  ...      2      0      8   
1               MUARO TAKUNG      0      0      0  ...      4      0      6   
2     KUNANGAN PARIT RANTANG      0      0      0  ...      9      2     17   
3                     KAMANG      1      1      0  ...      8      0      7   
4                    AIR AMO      0      0      0  ...      0      0      2   

KBLI  96112  96121  96122  96129  96200  96990  jumlah_usaha  
0         2      2      0      0     11      4           904  
1         7      0      0 

In [5]:
result.to_excel("DataReferenceFinal.xlsx", index=False)

## Joining Data Hasil Cleaning dengan File Geometry dan Prelist SE

In [7]:
import geopandas as gpd
import pandas as pd

# =====================================================
# 1️⃣ LOAD DATA
# =====================================================

gdf = gpd.read_file("Kelurahan_Sijunjung.shp")

df_ref_final = pd.read_excel("DataReferenceFinal.xlsx", sheet_name="Reference")
df_ref = pd.read_excel("DataReferenceFinal.xlsx", sheet_name="Utama")

# =====================================================
# 2️⃣ STANDARISASI NAMA KOLOM
# =====================================================

for df in [gdf, df_ref_final, df_ref]:
    df.columns = df.columns.str.strip().str.lower()

# =====================================================
# 3️⃣ DROP KOLOM TIDAK DIPERLUKAN
# =====================================================

cols_to_drop = [
    "kdprov",
    "kdkab",
    "kdebps",
    "kdpbps"
]

gdf = gdf.drop(columns=[col for col in cols_to_drop if col in gdf.columns])

print("Kolom setelah drop:")
print(gdf.columns.tolist())

# =====================================================
# 4️⃣ BERSIHKAN & SAMAKAN TIPE DATA
# =====================================================

gdf["kdepum"] = gdf["kdepum"].astype(str).str.strip()
df_ref_final["kdepum"] = df_ref_final["kdepum"].astype(str).str.strip()

df_ref_final["wadmkd"] = df_ref_final["wadmkd"].astype(str).str.strip()
df_ref["nmdesa"] = df_ref["nmdesa"].astype(str).str.strip()

# =====================================================
# 5️⃣ JOIN 1 (kdepum → wadmkd)
# =====================================================

if "wadmkd" in gdf.columns:
    gdf = gdf.drop(columns=["wadmkd"])

gdf_join1 = gdf.merge(
    df_ref_final[["kdepum", "wadmkd"]].drop_duplicates(),
    on="kdepum",
    how="left"
)

print("Join pertama selesai")

# =====================================================
# 6️⃣ JOIN 2 (wadmkd ↔ nmdesa)
# =====================================================

gdf_final = gdf_join1.merge(
    df_ref,
    left_on="wadmkd",
    right_on="nmdesa",
    how="left"
)

print("Join kedua selesai")

# =====================================================
# 7️⃣ VALIDASI
# =====================================================

print("Jumlah baris:", len(gdf_final))
print("Missing geometry:", gdf_final.geometry.isna().sum())

# Pastikan tetap GeoDataFrame
gdf_final = gpd.GeoDataFrame(gdf_final, geometry="geometry", crs=gdf.crs)

Kolom setelah drop:
['kdepum', 'kdpkab', 'kdppum', 'luaswh', 'tipadm', 'wadmkc', 'wadmkd', 'wadmkk', 'wadmpr', 'luas', 'geometry']
Join pertama selesai
Join kedua selesai
Jumlah baris: 62
Missing geometry: 0


In [8]:
# =====================================================
# 8️⃣ CONVERT CRS UNTUK WEB
# =====================================================

gdf_final = gdf_final.to_crs(epsg=4326)

# =====================================================
# 9️⃣ SAVE GEOJSON
# =====================================================

output_geojson = "DataRef_Final.geojson"
gdf_final.to_file(output_geojson, driver="GeoJSON")

print("GeoJSON berhasil dibuat:", output_geojson)

GeoJSON berhasil dibuat: DataRef_Final.geojson


## Reduksi File json

In [2]:
import pandas as pd
import json
import re

# ==============================
# FILE PATH
# ==============================
excel_file = "DataReferenceFinal.xlsx"
json_file = "kbli2025Lengkap.json"

# ==============================
# LOAD EXCEL
# ==============================
df = pd.read_excel(excel_file, sheet_name="Utama")

# ==============================
# AMBIL KODE KBLI DARI HEADER
# ==============================
kbli_used = []

for col in df.columns:
    
    col_str = str(col).strip()

    # hanya kode 5 digit
    if re.fullmatch(r"\d{5}", col_str):
        kbli_used.append(col_str)

print("Jumlah KBLI dipakai di Excel:", len(kbli_used))

# ==============================
# LOAD JSON KBLI
# ==============================
with open(json_file, "r", encoding="utf-8") as f:
    kbli_reference = json.load(f)

# ==============================
# FILTER JSON
# ==============================
filtered_kbli = {}

for kode in kbli_used:
    if kode in kbli_reference:
        filtered_kbli[kode] = kbli_reference[kode]

print("Jumlah KBLI setelah filter:", len(filtered_kbli))

# ==============================
# SAVE JSON BARU
# ==============================
output_file = "kbli2025_Filtered.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(filtered_kbli, f, indent=2, ensure_ascii=False)

print("JSON baru berhasil dibuat:", output_file)

Jumlah KBLI dipakai di Excel: 792
Jumlah KBLI setelah filter: 791
JSON baru berhasil dibuat: kbli2025_Filtered.json


## Join Excel ke JSON

In [1]:
import pandas as pd
import json

# =========================
# 1. Load Excel
# =========================
excel_path = "kbliUpdated.xlsx"
df = pd.read_excel(excel_path, dtype={"kodeKBLI": str})

# =========================
# 2. Load JSON lama
# =========================
json_path = "kbli2025.json"

with open(json_path, "r", encoding="utf-8") as f:
    kbli_json = json.load(f)

# =========================
# 3. Filter yang ada keterangannya
# =========================
df = df.dropna(subset=["Keterangan"])

# =========================
# 4. Tambahkan ke JSON
# =========================
for _, row in df.iterrows():
    kode = row["kodeKBLI"]
    ket = row["Keterangan"]

    kbli_json[kode] = ket

# =========================
# 5. Simpan JSON baru
# =========================
with open("kbli2025Lengkap.json", "w", encoding="utf-8") as f:
    json.dump(kbli_json, f, indent=4, ensure_ascii=False)

print("Selesai. JSON sudah diperbarui.")

Selesai. JSON sudah diperbarui.


## Join File Geojson dan Excel untuk Data Tambahan

In [7]:
import json
import pandas as pd

# =========================
# LOAD DATA
# =========================
geojson_path = "DataRef_Final.geojson"
excel_path = "dataNagari.xlsx"

with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

df_excel = pd.read_excel(excel_path)

# =========================
# NORMALISASI NAMA DESA
# =========================
df_excel["nmdesa"] = df_excel["nmdesa"].astype(str).str.strip().str.upper()

for feature in geojson_data["features"]:
    if "nmdesa" in feature["properties"]:
        feature["properties"]["nmdesa"] = str(feature["properties"]["nmdesa"]).strip().upper()

# =========================
# CEK DUPLIKAT DI EXCEL
# =========================
dup_count = df_excel["nmdesa"].value_counts()
dup_count = dup_count[dup_count > 1]

if not dup_count.empty:
    print("\n=== NAMA DESA DUPLIKAT DI EXCEL ===")
    print(dup_count)

    print("\n=== BARIS DATA DUPLIKAT ===")
    duplicates = df_excel[df_excel["nmdesa"].isin(dup_count.index)]
    print(duplicates.sort_values("nmdesa"))

    # Jika ingin otomatis digabung
    print("\nMenggabungkan data duplikat dengan SUM...\n")
    df_excel = df_excel.groupby("nmdesa", as_index=False).agg({
        "jml_jorong": "sum",
        "jml_pdd": "sum"
    })
else:
    print("\nTidak ada duplikat nmdesa di Excel\n")

# =========================
# CEK DESA DI GEOJSON
# =========================
desa_geojson = set(
    feature["properties"].get("nmdesa")
    for feature in geojson_data["features"]
)

desa_excel = set(df_excel["nmdesa"])

# Desa yang ada di Excel tapi tidak ada di GeoJSON
missing_geojson = desa_excel - desa_geojson

# Desa yang ada di GeoJSON tapi tidak ada di Excel
missing_excel = desa_geojson - desa_excel

print("\n=== DESA DI EXCEL TIDAK ADA DI GEOJSON ===")
print(missing_geojson)

print("\n=== DESA DI GEOJSON TIDAK ADA DI EXCEL ===")
print(missing_excel)

# =========================
# BUAT LOOKUP UNTUK JOIN
# =========================
lookup = df_excel.set_index("nmdesa")[["jml_jorong", "jml_pdd"]].to_dict("index")

# =========================
# JOIN KE GEOJSON
# =========================
join_count = 0

for feature in geojson_data["features"]:
    desa = feature["properties"].get("nmdesa")

    if desa in lookup:
        feature["properties"]["jml_jorong"] = int(lookup[desa]["jml_jorong"])
        feature["properties"]["jml_pdd"] = int(lookup[desa]["jml_pdd"])
        join_count += 1

print(f"\nJumlah feature berhasil dijoin: {join_count}")

from shapely.geometry import shape

# =========================
# HITUNG LUAS WILAYAH
# =========================
TOTAL_AREA_KM2 = 3150.8

# hitung luas geometri mentah
areas = []
for feature in geojson_data["features"]:
    geom = shape(feature["geometry"])
    areas.append(geom.area)

total_geom_area = sum(areas)

print("\nTotal area geometri (unit geojson):", total_geom_area)

# hitung rasio dan luas wilayah
for i, feature in enumerate(geojson_data["features"]):
    ratio = areas[i] / total_geom_area
    luas = ratio * TOTAL_AREA_KM2

    feature["properties"]["luas_wilayah"] = round(luas, 2)

print("Kolom luas_wilayah berhasil ditambahkan")

# =========================
# SIMPAN GEOJSON BARU
# =========================
output_path = "DataDashboardFinal.geojson"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(geojson_data, f, ensure_ascii=False, indent=2)

print("\nJoin selesai, file tersimpan di:", output_path)


Tidak ada duplikat nmdesa di Excel


=== DESA DI EXCEL TIDAK ADA DI GEOJSON ===
set()

=== DESA DI GEOJSON TIDAK ADA DI EXCEL ===
set()

Jumlah feature berhasil dijoin: 62

Total area geometri (unit geojson): 0.2559749528474825
Kolom luas_wilayah berhasil ditambahkan

Join selesai, file tersimpan di: DataDashboardFinal.geojson
